# 01 — Preparação dos Dados e EDA

Lê os parquets originais do **Amazon Shopping Queries Dataset (ESCI)**, filtra o locale (catálogo completo; `N_PRODUCTS` permite amostrar),
gera `data/processed/products_us.parquet` e `examples_us.parquet` e faz a análise exploratória.
Este notebook substitui o antigo `scripts/01_prepare_data.py`.

**Entrada:** os arquivos abaixo são baixados automaticamente para `data/raw/` (seção 1), ou copie-os manualmente:
- `shopping_queries_dataset_products.parquet`
- `shopping_queries_dataset_examples.parquet`

Para alternar CPU/GPU, altere apenas `DEVICE` na célula de setup.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

# ── Configuração única ───────────────────────────────────────────────
DEVICE = "auto"                     # "cpu" | "cuda" | "auto"
DRIVE_PROJECT_DIR = "MyDrive/tcc"   # pasta do projeto no Google Drive (só Colab)
# ─────────────────────────────────────────────────────────────────────

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive") / DRIVE_PROJECT_DIR
    %pip install -q polars pyarrow matplotlib
else:
    PROJECT_ROOT = Path.cwd()

assert (PROJECT_ROOT / "src").exists(), f"Pasta 'src' não encontrada em {PROJECT_ROOT}"
sys.path.insert(0, str(PROJECT_ROOT))

from src.device import build_runtime_config, collect

RUNTIME = build_runtime_config(DEVICE)
print(f"Projeto : {PROJECT_ROOT}")
print(f"Colab   : {IN_COLAB}")
print(f"Device  : {RUNTIME.device} (batch_size={RUNTIME.batch_size}, max_seq_length={RUNTIME.max_seq_length})")

In [ ]:
import matplotlib.pyplot as plt
import polars as pl
import pyarrow.parquet as pq

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "results" / "report"
for directory in (PROCESSED_DIR, REPORT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

LOCALE = "us"
N_PRODUCTS: int | None = None   # None = catálogo completo do locale; um inteiro amostra esse número de produtos
SEED = 42

PRODUCTS_FILE = RAW_DIR / "shopping_queries_dataset_products.parquet"
EXAMPLES_FILE = RAW_DIR / "shopping_queries_dataset_examples.parquet"
EXAMPLE_COLUMNS = ["query_id", "query", "product_id", "esci_label", "split"]

plt.rcParams["figure.dpi"] = 120

## 1. Download dos dados

Baixa do repositório oficial ([esci-data](https://github.com/amazon-science/esci-data/tree/main/shopping_queries_dataset)) os dois parquets (cerca de 1,1 GB no total) e o CSV de fontes para `data/raw/`. Arquivos já baixados com o tamanho correto são pulados. Defina `DOWNLOAD_DATA = False` se você já copiou os arquivos manualmente.

In [ ]:
from src.download import download_esci

DOWNLOAD_DATA = True         # False se os arquivos já estiverem em data/raw/
OVERWRITE_DOWNLOAD = False   # True baixa novamente mesmo que o arquivo exista

if DOWNLOAD_DATA:
    downloaded = download_esci(RAW_DIR, overwrite=OVERWRITE_DOWNLOAD)
    print("\nArquivos em data/raw/:", *(p.name for p in downloaded), sep="\n  ")

## 2. Verificação dos arquivos de entrada

In [ ]:
missing = [p for p in (PRODUCTS_FILE, EXAMPLES_FILE) if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Arquivos não encontrados:\n  " + "\n  ".join(str(p) for p in missing)
        + "\nUse DOWNLOAD_DATA = True na seção 1 ou copie os arquivos para data/raw/."
    )

for path in (PRODUCTS_FILE, EXAMPLES_FILE):
    meta = pq.ParquetFile(path).metadata
    print(f"{path.name}: {meta.num_rows:,} linhas | {meta.num_columns} colunas")

## 3. Produtos: filtro de locale e amostragem

In [ ]:
products_lf = pl.scan_parquet(PRODUCTS_FILE).filter(pl.col("product_locale") == LOCALE)
products_locale = collect(products_lf, RUNTIME.device)
print(f"Produtos no locale '{LOCALE}': {products_locale.height:,}")

products = (
    products_locale
    if N_PRODUCTS is None
    else products_locale.sample(n=min(N_PRODUCTS, products_locale.height), seed=SEED)
)
missing_title = products["product_title"].null_count()
print(f"Catálogo: {products.height:,} produtos | únicos: {products['product_id'].n_unique():,}")
print(f"Sem título: {missing_title:,} ({100 * missing_title / products.height:.1f}%)")

## 4. Examples: filtro de locale e junção com o catálogo amostrado

In [ ]:
examples_lf = pl.scan_parquet(EXAMPLES_FILE)
schema_names = examples_lf.collect_schema().names()

locale_col = next((c for c in ("query_locale", "product_locale") if c in schema_names), None)
if locale_col is None:
    print("AVISO: coluna de locale não encontrada em examples; mantendo todos os registros.")
else:
    examples_lf = examples_lf.filter(pl.col(locale_col) == LOCALE)

examples = collect(
    examples_lf.join(products.lazy().select("product_id"), on="product_id", how="inner").select(EXAMPLE_COLUMNS),
    RUNTIME.device,
)
print(f"Examples após filtro e junção: {examples.height:,} | queries: {examples['query_id'].n_unique():,}")

orphans = examples.join(products.select("product_id"), on="product_id", how="anti").height
assert orphans == 0, f"{orphans} examples referenciam produtos fora do catálogo"

## 5. Salvando os arquivos processados

In [ ]:
products_path = PROCESSED_DIR / f"products_{LOCALE}.parquet"
examples_path = PROCESSED_DIR / f"examples_{LOCALE}.parquet"

products.write_parquet(products_path, compression="zstd")
examples.write_parquet(examples_path, compression="zstd")

for path in (products_path, examples_path):
    meta = pq.ParquetFile(path).metadata
    print(f"{path.name}: {meta.num_rows:,} linhas | {path.stat().st_size / 1024**2:.1f} MB")

## 6. EDA — distribuição por split

In [ ]:
split_dist = (
    examples.group_by("split")
    .agg(n_pairs=pl.len(), n_queries=pl.col("query_id").n_unique())
    .with_columns(pct_pairs=(pl.col("n_pairs") / pl.col("n_pairs").sum() * 100).round(1))
    .sort("split")
)
split_dist

## 7. EDA — distribuição dos labels ESCI

In [ ]:
LABEL_NAMES = {"E": "Exact", "S": "Substitute", "C": "Complement", "I": "Irrelevant"}

label_dist = (
    examples.group_by("esci_label")
    .agg(n=pl.len())
    .with_columns(pct=(pl.col("n") / pl.col("n").sum() * 100).round(2))
    .sort("n", descending=True)
)
print(label_dist)

fig, ax = plt.subplots(figsize=(6, 3.5))
labels = [LABEL_NAMES.get(v, v) for v in label_dist["esci_label"]]
bars = ax.bar(labels, label_dist["pct"].to_list(), color="#1f77b4")
ax.bar_label(bars, fmt="%.1f%%")
ax.set_ylabel("% dos pares")
ax.set_title("Distribuição dos labels ESCI")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(REPORT_DIR / "eda_label_distribution.png", bbox_inches="tight")
plt.show()

## 8. EDA — julgamentos por query

In [ ]:
per_query = examples.group_by("query_id").agg(n=pl.len())
print(per_query["n"].describe())

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(per_query["n"].to_numpy(), bins=50, color="#2ca02c")
ax.set_xlabel("Produtos julgados por query")
ax.set_ylabel("Queries")
ax.set_title("Julgamentos por query")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(REPORT_DIR / "eda_judgments_per_query.png", bbox_inches="tight")
plt.show()

## 9. EDA — qualidade do catálogo

In [ ]:
null_rates = (
    pl.DataFrame({"column": products.columns, "nulls": products.null_count().row(0)})
    .with_columns(pct=(pl.col("nulls") / products.height * 100).round(1))
    .sort("pct", descending=True)
)
print(null_rates)

title_len = products.select(pl.col("product_title").str.len_chars().alias("chars")).drop_nulls()
print(title_len["chars"].describe())

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(title_len["chars"].to_numpy(), bins=60, color="#ff7f0e")
ax.set_xlabel("Caracteres no título")
ax.set_ylabel("Produtos")
ax.set_title("Comprimento dos títulos")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(REPORT_DIR / "eda_title_length.png", bbox_inches="tight")
plt.show()

## 10. EDA — campos de texto dos produtos

O comprimento dos campos orienta a escolha do texto usado para gerar os embeddings.

In [ ]:
TEXT_FIELDS = ["product_title", "product_description", "product_bullet_point", "product_brand"]

field_stats = pl.DataFrame(
    [
        {
            "campo": field,
            "preenchido_%": round(100 * (1 - products[field].null_count() / products.height), 1),
            **products.select(pl.col(field).str.len_chars().alias("n")).drop_nulls().select(
                media=pl.col("n").mean().round(0),
                mediana=pl.col("n").median(),
                p95=pl.col("n").quantile(0.95),
                max=pl.col("n").max(),
            ).row(0, named=True),
        }
        for field in TEXT_FIELDS
    ]
)
field_stats

## 11. Estimativa de truncamento por modelo

Estima tokens como caracteres / 4 (aproximação para inglês) sobre `product_title`, o único campo que o
indexador concatena atualmente (`_EMBEDDING_FIELDS` em `src/elasticsearch/indexer.py`).

In [ ]:
from src.models.registry import list_models

CHARS_PER_TOKEN = 4
est_tokens = products.select(
    (pl.col("product_title").fill_null("").str.len_chars() / CHARS_PER_TOKEN).alias("tokens")
)["tokens"]

print(f"Tokens estimados: média={est_tokens.mean():.0f} | mediana={est_tokens.median():.0f} | "
      f"p95={est_tokens.quantile(0.95):.0f} | máx={est_tokens.max():.0f}\n")
for cfg in list_models():
    truncated = (est_tokens > cfg.max_seq_length).mean() * 100
    print(f"{cfg.display_name:<18} (max={cfg.max_seq_length:>5}): {truncated:5.1f}% truncados")

## 12. EDA — queries

In [ ]:
query_df = examples.unique(subset="query_id").select(
    "query_id", "query", n_words=pl.col("query").str.split(" ").list.len()
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(query_df["n_words"].to_numpy(), bins=30, color="#9467bd")
axes[0].set_xlabel("Palavras por query")
axes[0].set_ylabel("Queries")
axes[0].set_title("Comprimento das queries")
axes[1].hist(per_query["n"].to_numpy(), bins=40, color="#8c564b")
axes[1].set_xlabel("Produtos por query")
axes[1].set_ylabel("Queries")
axes[1].set_title("Produtos candidatos por query")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(REPORT_DIR / "eda_query_stats.png", bbox_inches="tight")
plt.show()

print(f"Palavras por query: média={query_df['n_words'].mean():.1f} | mediana={query_df['n_words'].median():.0f} | máx={query_df['n_words'].max()}")

In [ ]:
# Queries do split de teste com mais produtos julgados
(
    examples.filter(pl.col("split") == "test")
    .group_by("query_id", "query")
    .agg(
        n_products=pl.len(),
        n_exact=(pl.col("esci_label") == "E").sum(),
        n_irrelevant=(pl.col("esci_label") == "I").sum(),
    )
    .sort("n_products", descending=True)
    .head(10)
)

## 13. Resumo para o TCC

In [ ]:
train_df = examples.filter(pl.col("split") == "train")
test_df = examples.filter(pl.col("split") == "test")

print("=" * 50)
print(f"RESUMO DO DATASET (locale={LOCALE})")
print("=" * 50)
print(f"Produtos únicos        : {products['product_id'].n_unique():>10,}")
print(f"Queries únicas (total) : {examples['query_id'].n_unique():>10,}")
print(f"  treino               : {train_df['query_id'].n_unique():>10,}")
print(f"  teste                : {test_df['query_id'].n_unique():>10,}")
print(f"Pares (query, produto) : {examples.height:>10,}")
print("\nDistribuição ESCI (teste):")
for label, name in LABEL_NAMES.items():
    n = test_df.filter(pl.col("esci_label") == label).height
    print(f"  {name:<11} ({label}): {n:>8,} ({100 * n / test_df.height:.1f}%)")
print(f"\nComprimento médio do título: {products['product_title'].str.len_chars().mean():.0f} caracteres")
print(f"Produtos sem descrição     : {100 * products['product_description'].null_count() / products.height:.1f}%")
print(f"Produtos sem brand         : {100 * products['product_brand'].null_count() / products.height:.1f}%")